# Topic: SQL Subquery Pattern (Scalar, Correlated, Derived, IN vs EXISTS)

## Definition (30-second explanation)
A subquery (or nested query) is a SQL query embedded within another SQL query, allowing you to break complex problems into smaller steps by using the result of one query as input to another. They can be placed in the `SELECT`, `FROM`, `WHERE`, and `HAVING` clauses. 

## Why Interviewers Ask This
Interviewers test subqueries to evaluate your ability to think relationally, handle complex filtering, and understand database performance implications. They specifically look for your awareness of common pitfalls like the "NULL trap" and knowing when to optimize a correlated subquery into a `JOIN` or Window Function.

## Core Concepts
*   **Scalar Subquery:** Returns exactly one value (1 row, 1 column) and is typically used in `SELECT` or `WHERE` clauses. Must use `LIMIT 1` if uniqueness isn't guaranteed.
*   **Correlated Subquery:** References columns from the outer query and executes once for every row processed by the outer query (O(n) executions).
*   **Derived Table:** A subquery located in the `FROM` clause that acts as a temporary table. It **must** be given an alias, even if unused.
*   **IN vs EXISTS:** `IN` filters by a list of values, while `EXISTS` evaluates to TRUE/FALSE if any matching record exists. 

## When to Use
*   Use a **Scalar Subquery** to compare individual row values against an aggregate metric (e.g., comparing an employee's salary to the company average).
*   Use a **Derived Table** to pre-aggregate or filter data before joining it to other large tables.
*   Use **EXISTS** to check for the presence of related records, especially in large datasets where short-circuiting improves performance.

## Advantages
*   Improves query readability by breaking complex logic into manageable, logical steps.
*   `EXISTS` and `NOT EXISTS` handle `NULL` values correctly and stop processing at the first match, making them highly efficient.
*   Derived tables can act similarly to CTEs, allowing complex multi-step transformations without creating actual views.

## Limitations
*   **Correlated Subqueries** can be extremely slow because they execute repeatedly for every row in the outer query.
*   Subqueries inside `IN` clauses can suffer from poor performance on very large lists compared to `EXISTS` or `JOIN`s.

## Common Comparisons
*   **WHERE IN vs EXISTS:** `EXISTS` is faster for large lists as it stops at the first match and handles NULLs safely; `IN` is slower and ignores NULLs.
*   **WHERE NOT IN vs NOT EXISTS:** `NOT EXISTS` is the safest way to find non-matching rows; `NOT IN` is a trap because if the subquery returns *any* NULL, the entire result is empty (always FALSE).
*   **Correlated Subquery vs Window Function:** Correlated subqueries run per row and are slow; Window functions compute aggregates over partitions simultaneously and are preferred for performance.

## Common Interview Traps
*   **The NOT IN + NULL Trap:** Writing `WHERE id NOT IN (SELECT id...)` without filtering NULLs in the subquery.
*   **Missing Derived Table Alias:** Forgetting to name a subquery in the `FROM` clause, which throws an immediate syntax error.
*   **Scalar Multi-Row Error:** Writing a scalar subquery that accidentally returns multiple rows, breaking the query.

## Python / SQL Syntax
```sql
-- Derived Table (Must have alias)
SELECT dept_stats.department, dept_stats.avg_salary
FROM (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
) AS dept_stats;

-- Correlated Subquery
SELECT emp_name, salary
FROM employees e1
WHERE salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.department = e1.department
);
```

## 45-Second Interview Answer
Subqueries are queries nested within another query to simplify complex filtering or aggregation. A scalar subquery returns a single value, often used to compare a row against an aggregate. A derived table lives in the `FROM` clause to pre-aggregate data and must be aliased. The biggest interview pitfalls are correlated subqueries, which run once per row and can cause performance bottlenecks, and the `NOT IN` operator, which silently fails and returns nothing if the subquery contains a `NULL` value. I generally prefer `NOT EXISTS` or `LEFT JOIN` with a `NULL` check to safely avoid this trap.

## Example Questions:

### Q1: Find all customers who placed orders in January 2024 but NOT in February 2024.

**Ideal Interview Answer (MySQL):**
```sql
SELECT DISTINCT customer_id 
FROM orders 
WHERE order_date >= '2024-01-01' AND order_date < '2024-02-01'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o2 
      WHERE o2.customer_id = orders.customer_id 
        AND o2.order_date >= '2024-02-01' AND o2.order_date < '2024-03-01'
  );
```

**Common Mistakes Candidates Make:**
*   Using `NOT IN` without ensuring the subquery explicitly filters out `NULL` customer IDs.
*   Using functions on the indexed column (e.g., `MONTH(order_date) = 1`) which breaks SARGability (index usage).

**Likely Interviewer Follow-up:**
"How would you rewrite this exact query using a `LEFT JOIN` instead of `NOT EXISTS`?"

### Q2: Write a correlated subquery to find each product's price compared to its category's average price.

**Ideal Interview Answer (MySQL):**
```sql
SELECT 
    product_id, 
    price, 
    (SELECT AVG(price) 
     FROM products p2 
     WHERE p2.category_id = p1.category_id) AS category_avg_price
FROM products p1;
```

**Common Mistakes Candidates Make:**
*   Forgetting to alias the tables (`p1`, `p2`) to establish the correlation correctly.
*   Accidentally returning multiple columns or rows in the `SELECT` clause subquery, which throws a scalar subquery error.

**Likely Interviewer Follow-up:**
"This correlated subquery runs O(N) times. How can you rewrite this to be more performant using modern SQL features?" (Answer: Window Functions).

### Q3: Find the department with the highest average salary using only subqueries (no window functions).

**Ideal Interview Answer (MySQL):**
```sql
SELECT department
FROM (
    SELECT department, AVG(salary) AS avg_sal
    FROM employees
    GROUP BY department
) AS dept_avgs
ORDER BY avg_sal DESC
LIMIT 1;
```

**Common Mistakes Candidates Make:**
*   Trying to use `MAX(AVG(salary))`, which is invalid syntax in SQL.
*   Forgetting the alias `AS dept_avgs` for the derived table in the `FROM` clause.

**Likely Interviewer Follow-up:**
"If two departments are tied for the exact same highest average salary, this query only returns one. How would you modify it using subqueries to return *both* tied departments?"

### Q4: Convert this subquery into a CTE: SELECT * FROM (SELECT dept, AVG(salary) avg_sal FROM emp GROUP BY dept) t WHERE t.avg_sal > 70000.

**Ideal Interview Answer (MySQL):**
```sql
WITH DeptAverages AS (
    SELECT dept, AVG(salary) AS avg_sal
    FROM emp
    GROUP BY dept
)
SELECT * 
FROM DeptAverages 
WHERE avg_sal > 70000;
```

**Common Mistakes Candidates Make:**
*   Adding an alias to the CTE call in the main query like a derived table (e.g., `FROM DeptAverages t`), which isn't strictly necessary but sometimes causes confusion.
*   Forgetting the `AS` keyword after the CTE name.

**Likely Interviewer Follow-up:**
"In what scenario would a CTE be functionally superior to a derived table?" (Answer: When you need to reference the same aggregated temporary result multiple times in the main query).

## Practice Questions:

### Q1:
**Scenario**
You are a Data Scientist for a SaaS company. The marketing team wants to send a promotional email to all registered users who have never made a purchase.

**Mock Schema**
```sql
CREATE TABLE users (
    user_id INT,
    user_name VARCHAR(50)
);

CREATE TABLE purchases (
    purchase_id INT,
    user_id INT,
    amount DECIMAL(10,2)
);

INSERT INTO users (user_id, user_name) VALUES 
(1, 'Alice'), (2, 'Bob'), (3, 'Charlie'), (4, 'David');

-- Notice the NULL user_id (e.g., a guest checkout system bug)
INSERT INTO purchases (purchase_id, user_id, amount) VALUES 
(101, 1, 50.00), (102, 2, 75.00), (103, NULL, 20.00);
```

**A junior analyst wrote the following query to get the list of users to email:**
```sql
SELECT user_name 
FROM users 
WHERE user_id NOT IN (SELECT user_id FROM purchases);
```

**Your Task:**

- What exactly will this query return given the mock data, and why?

- Rewrite the query so that it successfully returns the correct users (Charlie and David) using NOT EXISTS.


**Answer:**
Because there is a `NULL` value in the subquery's result set, the `NOT IN` evaluation will yield `UNKNOWN` for every row that doesn't match. In SQL, `WHERE UNKNOWN` filters out the row, resulting in an entirely empty result set.

To fix this and correctly return users without purchases, I would use `NOT EXISTS` which handles NULLs correctly:
```sql
SELECT user_name
FROM users u
WHERE NOT EXISTS (
    SELECT 1 
    FROM purchases p
    WHERE p.user_id = u.user_id
);
```

Alternatively, if I must use `NOT IN`
```sql
SELECT user_name 
FROM users 
WHERE user_id NOT IN (SELECT user_id FROM purchases where user_id is not null);
```
**Interview Tips:**
*   **The Golden Rule:** Never use `NOT IN` unless you are 100% certain the subquery cannot return a `NULL` value, or you explicitly add `WHERE column IS NOT NULL`.
*   **Performance:** For large datasets, `NOT EXISTS` is generally faster than `NOT IN` because it short-circuits (stops evaluating) as soon as it finds a matching row, whereas `NOT IN` must evaluate the entire list.

### Q2: Question 2 (Coding & Optimization): Eliminating Correlated Subqueries

**You are given an employees table:**
```sql
CREATE TABLE employees (
    emp_id INT,
    emp_name VARCHAR(50),
    department VARCHAR(50),
    salary DECIMAL(10,2)
);

INSERT INTO employees (emp_id, emp_name, department, salary) VALUES
(1, 'Alice', 'Engineering', 90000),
(2, 'Bob', 'Engineering', 120000),
(3, 'Charlie', 'Sales', 60000),
(4, 'David', 'Sales', 85000),
(5, 'Eve', 'HR', 70000);
```

**An analyst wrote this correlated subquery to find employees who earn more than the average salary in their specific department:**
```sql
SELECT emp_name, department, salary
FROM employees e1
WHERE salary > (
    SELECT AVG(salary) 
    FROM employees e2 
    WHERE e1.department = e2.department
);
```

**Your Task:**
The engineering team rejected this code in a pull request because correlated subqueries execute row-by-row (O(N) performance). Rewrite this exact logic to be highly performant using a Window Function.

**Answer:**
```sql
WITH DepartmentStats AS (
    SELECT 
        emp_id,
        emp_name, 
        department, 
        salary,
        AVG(salary) OVER(PARTITION BY department) AS dept_avg_sal
    FROM employees
) 
SELECT 
    emp_id, 
    emp_name, 
    department, 
    salary
FROM DepartmentStats
WHERE salary > dept_avg_sal;
```

**Interview Tips:**
*   **The "Why":** Correlated subqueries execute once for *every single row* in the outer query, leading to O(N) performance which crashes on large tables. Window functions compute the aggregate simultaneously across partitions in a single pass.
*   **The CTE Requirement:** You cannot put a Window Function directly into a `WHERE` clause (e.g., `WHERE salary > AVG(salary) OVER(...)`). It throws a syntax error. You *must* calculate it first in a CTE or Derived Table, and then filter on the alias in the outer query.

### Q3:
**Your Tasks:**

- Correlated Subquery: Write a query that returns every author's name and the total number of books they have written. You must use a correlated subquery in the SELECT clause to calculate the count (do not use JOIN and GROUP BY).

- NOT EXISTS: Write a query to find the names of authors who have never published a book (i.e., they do not exist in the books table). You must use NOT EXISTS.

**Mock Schema**
```sql
CREATE TABLE authors (
    author_id INT,
    author_name VARCHAR(50)
);

CREATE TABLE books (
    book_id INT,
    author_id INT,
    title VARCHAR(100)
);

INSERT INTO authors (author_id, author_name) VALUES
(1, 'J.K. Rowling'), 
(2, 'George R.R. Martin'), 
(3, 'Harper Lee'), 
(4, 'Unknown Author'); -- This author has no books

INSERT INTO books (book_id, author_id, title) VALUES
(101, 1, 'Harry Potter 1'),
(102, 1, 'Harry Potter 2'),
(103, 2, 'Game of Thrones'),
(104, 3, 'To Kill a Mockingbird');
```

**Answer:**
```sql
-- Task 1: Correlated Subquery in SELECT
SELECT 
    author_id, 
    author_name,
    (SELECT COUNT(*) 
     FROM books b 
     WHERE b.author_id = a.author_id) AS num_of_books
FROM authors a;

-- Task 2: NOT EXISTS
SELECT author_id, author_name
FROM authors a
WHERE NOT EXISTS (
    SELECT 1
    FROM books b
    WHERE b.author_id = a.author_id
);
```

**Interview Tips:**
*   **Correlated Subquery in SELECT:** This acts effectively like a `LEFT JOIN` combined with a `GROUP BY`. It ensures that authors with 0 books return `0` instead of dropping out of the result set (which would happen with a standard `INNER JOIN`).
*   **The SELECT 1 trick:** Inside an `EXISTS` or `NOT EXISTS` clause, what you `SELECT` doesn't matter (you can `SELECT *`, `SELECT 1`, `SELECT 'apple'`). The database engine only evaluates whether a row *exists* that satisfies the `WHERE` condition. `SELECT 1` is the universally accepted standard for readability.

### Q4:
**You have an orders table tracking purchases across your e-commerce platform.**

**Mock Schema**
```sql
CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    amount DECIMAL(10,2)
);

INSERT INTO orders (order_id, customer_id, order_date, amount) VALUES
(1, 101, '2023-01-15', 50.00),
(2, 101, '2023-03-22', 120.00),
(3, 102, '2023-02-10', 75.00),
(4, 102, '2023-02-14', 90.00),
(5, 103, '2023-01-05', 30.00);
```

**Your Task:**
Write a query to retrieve the entire row (order_id, customer_id, order_date, amount) for the most recent order placed by each customer.

*Constraint: You MUST use a Correlated Subquery in the WHERE clause to solve this. (Do not use Window Functions or GROUP BY for this specific exercise).*


**Answer:**
```sql
SELECT 
    order_id, 
    customer_id, 
    order_date, 
    amount
FROM orders o1
WHERE order_date = (
    SELECT MAX(order_date)
    FROM orders o2
    WHERE o2.customer_id = o1.customer_id
);
```

**Interview Tips:**
*   **The Logic:** For every row the engine looks at (o1), it pauses, runs the subquery (o2) to find the absolute max date for *that specific customer*, and then checks if o1's date matches that max date. 
*   **The Alternative (Window Function):** Interviewers will almost always ask how to optimize this. The modern equivalent is using a CTE with `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC)` and filtering for `row_num = 1`. 
*   **The Tie-Breaker:** Note that if a customer has two orders on their maximum date, this correlated subquery approach returns *both* rows.